# SQANTI3 rescue worksheet

In this final worksheet, we will go through the results obtained in the rescue module of SQANTI3. For this example, we have only run the full rescue using rules, since the automatic rescue is always run first. We will compare both outputs and see which isoforms were rescued, and why.

---

## 🤖 Automatic rescue 

For both parts of the exercise we will use the main rescue table `mouse_rescue_table.tsv` to see which transcripts were rescued in each mode, and all their respective information.

1. **How many transcripts were rescued automatically?**

In [3]:
suppressMessages(library(readr))
suppressMessages(library(dplyr))

rescue.df <- read_tsv("results/05_Rescue_full/human_chr8_rescue_table.tsv",show_col_types=F)

rescue.df %>%
    filter(rescue_mode == "automatic") %>%
    nrow()

[1] 235

<details><summary>Answer</summary><br>
235 transcripts
</details><br>

2. **From the reintroduced transcripts, how many associated isoforms they had in total?**

In [4]:
rescue.df %>%
    filter(rescue_mode == "automatic" ) %>%
    pull(assigned_transcript)  %>% unique()-> auto_reintroduced

complete.df <- read_tsv("results/04_Filter_orthogonal/human_chr8_RulesFilter_classification.txt",show_col_types=F)

complete.df %>%
    filter(associated_transcript %in% auto_reintroduced) %>%
    nrow()

[1] 239

<details><summary>Answer</summary><br>
The rescued transcripts had 239 associated isoforms.
</details><br>

3. **Can you see how many transcripts and genes are still lost due to all of their isoforms being artifacts?**

In [3]:
complete.df %>%
    filter(filter_result == "Artifact" & ! associated_transcript %in% auto_reintroduced) %>%
    filter(!stringr::str_detect(associated_gene,"novel")) %>%
    pull(associated_gene) %>%
    unique() %>% length()

complete.df %>%
    filter(filter_result == "Artifact" & ! associated_transcript %in% auto_reintroduced) %>%
    pull(associated_transcript) %>%
    unique() %>% length()

complete.df %>%
    filter(filter_result == "Artifact" & ! associated_transcript %in% auto_reintroduced) %>%
    filter(associated_transcript == "novel")

[1] 281
[1] 221
# A tibble: 466 × 61
   isoform           chrom  start    end strand length exons structural_category
   <chr>             <chr>  <dbl>  <dbl> <chr>   <dbl> <dbl> <chr>              
 1 ENSG00000008513.… chr8  1.33e8 1.33e8 -        2987     8 novel_in_catalog   
 2 ENSG00000008513.… chr8  1.33e8 1.33e8 -        3190     7 novel_in_catalog   
 3 ENSG00000008988.… chr8  5.61e7 5.61e7 -         521     4 novel_not_in_catal…
 4 ENSG00000023287.… chr8  5.26e7 5.27e7 -        6416    25 novel_in_catalog   
 5 ENSG00000040341.… chr8  7.34e7 7.37e7 -        2882    14 novel_in_catalog   
 6 ENSG00000040341.… chr8  7.34e7 7.37e7 -        2769    13 novel_in_catalog   
 7 ENSG00000040341.… chr8  7.34e7 7.37e7 -        2996    15 novel_in_catalog   
 8 ENSG00000040341.… chr8  7.34e7 7.37e7 -        2703    12 novel_in_catalog   
 9 ENSG00000040341.… chr8  7.34e7 7.37e7 -        3127    16 novel_in_catalog   
10 ENSG00000040341.… chr8  7.35e7 7.37e7 -        2667    13 novel_in_ca

<details><summary>Answer</summary><br>
There are 221 reference transcripts and 281 genes still lost after the automatic rescue. No novel transcripts were lost.
</details><br>

---

## :world_map:  Mapping candidates

4. **How many candidate and target transcripts have been selected?**

In [5]:
candidate.df <- read_tsv("results/05_Rescue_full/human_chr8_rescue_candidates.tsv",show_col_types=F)
target.df <- read_tsv("results/05_Rescue_full/human_chr8_rescue_targets.tsv",show_col_types=F)

nrow(candidate.df)
nrow(target.df)

[1] 272

[1] 2721

<details><summary>Answer</summary>
- Candidates: 272
- Targets: 2721
</details><br>

5. **From the targets, how many are from the long read transcriptome and how many are reference transcripts?** 
*Hint: long-read transcripts have a "_" in their name*

In [9]:
target.df %>% filter(stringr::str_detect(isoform,"_")) %>%
    nrow()

[1] 619

<details><summary>Answer</summary>
- Reference: 2.102
- Long-read transcriptome: 619
</details><br>

6. **What is the average number of mapping hits that the candidates have? And the maximum?**

In [10]:
mapping_hits.df <- read_tsv("results/05_Rescue_full/human_chr8_rescue_mapping_hits.tsv", show_col_types=F)

mapping_hits.df %>% group_by(rescue_candidate) %>%
 summarise(n=n()) %>%
 mutate(avg = mean(n),
        max = max(n)) %>%
    select(avg,max) %>% distinct()

avg,max
<dbl>,<int>
4.595588,7


<details><summary>Answer</summary>
On average, there are 4.60 hits per candidate, and the candidate that mapped to the most targets mapped against 7 targets.
</details><br>

## 🌕 Rescue by mapping (full)

7. **After the full rescue, how many isoforms are in the final transcriptome? How many were recovered in this last step?**

In [11]:
new_classification.df <- read_tsv("results/05_Rescue_full/human_chr8_rescued_classification.txt",show_col_types=F)
nrow(new_classification.df)

rescue.df %>% select(rescue_mode) %>%
    table()

[1] 1807

rescue_mode
    automatic rules_mapping 
          235           372 

<details><summary>Answer</summary>
In total, we have 1807 isoforms after rescue. 235 come from the automatic rescue and 372 come from the full rescue (rules mapping).
</details><br>

8. **Can you find any rescued isoform that comes from the long-reads data?**

In [12]:
rescue.df %>% filter(rescue_mode == "rules_mapping" & origin == "lr_defined") %>%
    nrow()

[1] 287

<details><summary>Answer</summary>
In total, there are 287 rules mapping rescued isoforms from the `lr_defined` origin (long-read defined isoforms).
</details><br>

## Requantification

9. **How many counts were redistributed during the requantification step?**

In [13]:
requantification.df <- read_tsv("results/05_Rescue_full/human_chr8_reassigned_counts_extended.tsv",show_col_types=F) %>%
  mutate(old_count = rowSums(across(starts_with("old_"))),
         new_count = rowSums(across(starts_with("new_"))))

requantification.df %>% filter(new_count == 0) %>%
    pull(old_count) %>% sum()
requantification.df %>% pull(new_count) %>% sum()

[1] 27644

[1] 177982

<details><summary>Answer</summary>
In total, 27644 counts were redistributed during the requantification step (out of 177982 total counts).
</details><br>

10. **How many artifact isoforms had their counts completely reassigned to rescued transcripts (lost all counts)?**

In [10]:
requantification.df %>%
    filter(old_count > 0 & new_count == 0) %>%
    nrow()

[1] 933


<details><summary>Answer</summary>
933 artifact isoforms lost all their counts during requantification, as they were redistributed to the rescued isoforms that better represent those reads.
</details><br>

11. **How many new entries are there in the requantification table?**

In [11]:
requantification.df %>%
    filter(old_count == 0 & new_count > 0) %>%
    nrow()

[1] 479


<details><summary>Answer</summary>
There are 479 new entries in the requantification table, corresponding to the reintroduced reference transcripts.
</details><br>